# NEXUS-AURORA: Data Preparation

Downloads FineWeb-Edu dataset, trains a SentencePiece BPE tokenizer (vocab=8192),
and writes pre-tokenized binary memmap files for training.

**Run this notebook once** before starting training.

In [ ]:
# Install dependencies (Kaggle)
# !pip install sentencepiece datasets tqdm pyyaml -q

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/nexus-lm')

import os
from pathlib import Path

OUTPUT_DIR = '/kaggle/working/data'
TOKENIZER_PATH = f'{OUTPUT_DIR}/tokenizer.model'
N_TOKENS = 1_500_000_000
VOCAB_SIZE = 8192

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# Train tokenizer on FineWeb-Edu sample
from data.prepare import train_tokenizer_on_dataset

if not Path(TOKENIZER_PATH).exists():
    train_tokenizer_on_dataset('fineweb_edu', TOKENIZER_PATH, vocab_size=VOCAB_SIZE)
else:
    print(f'Tokenizer already exists at {TOKENIZER_PATH}')

In [ ]:
# Prepare training and validation data
from data.prepare import prepare_dataset

train_path, val_path = prepare_dataset(
    dataset_name='fineweb_edu',
    output_dir=OUTPUT_DIR,
    tokenizer_path=TOKENIZER_PATH,
    n_tokens_approx=N_TOKENS,
    val_fraction=0.01,
)
print(f'Train: {train_path}')
print(f'Val:   {val_path}')

In [ ]:
# Verify data
import numpy as np

train = np.memmap(train_path, dtype=np.uint16, mode='r')
val = np.memmap(val_path, dtype=np.uint16, mode='r')
print(f'Train tokens: {len(train):,}')
print(f'Val tokens:   {len(val):,}')
print(f'Token range: [{train.min()}, {train.max()}] (vocab_size={VOCAB_SIZE})')